# W1D5: 이미지 연산

> 📖 원리 이해 (검색): **"opencv python image arithmetic operations tutorial"** → docs.opencv.org 또는 pyimagesearch.com 결과
> 📖 원리 이해 (검색): **"opencv bitwise and or not mask tutorial"** → docs.opencv.org 또는 pyimagesearch.com 결과
> 📋 파라미터 확인: [OpenCV 공식 - absdiff](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#ga6fef31bc8c4071cbc114a758a2b79c14)
> 📋 파라미터 확인: [OpenCV 공식 - addWeighted](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#gafafb2513349db3bcff51f54ee5592a19)
> 📋 파라미터 확인: [OpenCV 공식 - bitwise_and](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#ga60b4d04b251ba5eb1392c34425497e14)

---

## 🧠 원리 이해

### absdiff — 왜 절댓값 차이로 결함을 찾을 수 있나?

비유: 정상 제품 사진과 검사 대상 제품 사진을 겹쳐놓고 "달라진 부분"만 형광펜으로 칠하는 작업입니다.

**동작 원리:**

```
dst[i, j] = |src1[i, j] - src2[i, j]|
```

- 두 이미지가 같은 픽셀 → 차이 = 0 → 결과 이미지에서 검정(0)
- 두 이미지가 다른 픽셀 → 차이 > 0 → 결과 이미지에서 밝음
- **절댓값**: 방향에 관계없이 차이가 크면 밝게 표시

```
정상 이미지: [200, 200, 200, 200, 200]
결함 이미지: [200, 200,  50, 200, 200]
absdiff 결과:[  0,   0, 150,   0,   0]  ← 결함 위치만 밝게
```

**왜 일반 빼기(subtract) 대신 absdiff?**
- `subtract`는 음수가 나오면 0으로 잘림 (결함이 밝은 경우 놓침)
- `absdiff`는 밝든 어둡든 모든 차이를 감지

**제조 검사 핵심**: W6D4에서 배울 "차이 기반 검출(diff detect)"의 가장 기초 연산입니다.

---

### addWeighted — 두 이미지를 어떻게 자연스럽게 합성할까?

**수식:**

```
dst = alpha × src1 + beta × src2 + gamma
```

| 파라미터 | 역할 | 예시 |
|----------|------|------|
| `alpha` | src1 가중치 (0.0~1.0) | 0.7 → src1이 70% 기여 |
| `beta` | src2 가중치 | 0.3 → src2가 30% 기여 |
| `gamma` | 밝기 오프셋 | 0 → 조정 없음, 양수 → 전체 밝게 |

**결과 범위**: 계산 후 0~255로 클리핑됨

**제조 검사 연결**: 결함 감지 결과(빨간 마스크)를 원본 이미지 위에 반투명하게 오버레이할 때 사용합니다.

---

### bitwise 연산 — 마스크로 관심 영역만 추출하기

비유: 스텐실(구멍 뚫린 판)을 사용해 특정 모양만 색칠하는 것처럼, **마스크**로 원하는 영역만 이미지에서 추출합니다.

**각 연산의 동작 (픽셀 단위, 비트 수준):**

```
bitwise_AND:  dst[i,j] = src1[i,j] & src2[i,j]
  → 두 픽셀이 모두 밝을 때만 밝음
  → 마스크(흰=255, 검=0)와 AND → 흰 부분만 원본 유지, 나머지 검정

bitwise_OR:   dst[i,j] = src1[i,j] | src2[i,j]
  → 둘 중 하나라도 밝으면 밝음
  → 두 이진 마스크를 합칠 때 사용

bitwise_NOT:  dst[i,j] = ~src[i,j]  (= 255 - src[i,j])
  → 픽셀값 반전. 흰색 ↔ 검정
  → 마스크의 안쪽/바깥쪽을 뒤집을 때 사용
```

**제조 검사 핵심**: `threshold` → `bitwise_AND`로 결함 영역만 원본에서 추출하는 패턴이 W3부터 반복됩니다.

---

**한 줄 요약**: `absdiff` = 변화 감지 / `addWeighted` = 결과 오버레이 / `bitwise` = 마스크 기반 영역 추출

## 📌 오늘의 목표
- `absdiff`로 두 이미지의 차이를 계산해 변화 영역 감지하기
- `addWeighted`로 두 이미지를 가중치 블렌딩하기
- `bitwise_and/or/not`으로 마스크를 이용한 관심 영역 추출하기
- 정상 이미지 vs 결함 이미지 비교로 결함 위치를 자동 감지하는 기초 파이프라인 체험

## 🎯 배울 함수

### `cv2.absdiff(src1, src2)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `src1` | `ndarray` | 첫 번째 이미지 (그레이스케일 또는 BGR, 동일 shape/dtype 필요) |
| `src2` | `ndarray` | 두 번째 이미지 |
| 반환값 | `ndarray` | `|src1 - src2|` 픽셀별 절댓값 차이. dtype은 입력과 동일 |

> 두 이미지의 픽셀 차이를 절댓값으로 계산합니다. 같은 곳은 0(검정), 다른 곳은 밝게 표시됩니다.

---

### `cv2.addWeighted(src1, alpha, src2, beta, gamma)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `src1` | `ndarray` | 첫 번째 이미지 |
| `alpha` | `float` | src1의 가중치. 0.0~1.0 범위 |
| `src2` | `ndarray` | 두 번째 이미지 (src1과 동일 shape/dtype) |
| `beta` | `float` | src2의 가중치. 0.0~1.0 범위 |
| `gamma` | `float` | 전체 밝기 오프셋. 보통 0.0 |
| 반환값 | `ndarray` | `alpha×src1 + beta×src2 + gamma`, 결과는 0~255 클리핑 |

> `alpha + beta = 1.0`이 되면 두 이미지의 자연스러운 블렌딩. 합이 1을 넘으면 전체적으로 밝아짐.

---

### `cv2.bitwise_and(src1, src2, mask=None)`
### `cv2.bitwise_or(src1, src2, mask=None)`
### `cv2.bitwise_not(src, mask=None)`
| 파라미터 | 타입 | 설명 |
|----------|------|------|
| `src1`, `src2` | `ndarray` | 입력 이미지 (동일 shape/dtype) |
| `mask` | `uint8 ndarray` | 선택적. 255인 픽셀에만 연산 적용, 0인 픽셀은 결과도 0 |
| 반환값 | `ndarray` | 비트 단위 연산 결과 |

> `mask` 파라미터로 관심 영역(ROI)만 선택적으로 연산할 수 있습니다. 이진 마스크(흰=255, 검=0)와 `bitwise_and`를 결합해 영역 추출하는 패턴이 핵심.

## 📦 import + 데이터 로딩

이미지 연산은 **두 이미지**를 다루므로, 같은 결함 유형에서 서로 다른 이미지를 로딩해 비교합니다.

- `_1.jpg`: 비교 기준이 될 "참조 이미지"
- `_2.jpg`: 비교 대상 "검사 이미지"

같은 유형이지만 결함 모양이 다른 두 이미지를 absdiff로 비교해 "변화 영역"을 감지하는 기초를 체험합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path("../data/kaggle/archive/NEU-DET/train/images")
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

img_ref  = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_1.jpg"), cv2.IMREAD_GRAYSCALE)
img_test = cv2.imread(str(DATA_DIR / "inclusion" / "inclusion_2.jpg"), cv2.IMREAD_GRAYSCALE)

print(f"참조 이미지: shape={img_ref.shape}, dtype={img_ref.dtype}, 범위={img_ref.min()}~{img_ref.max()}")
print(f"검사 이미지: shape={img_test.shape}, dtype={img_test.dtype}, 범위={img_test.min()}~{img_test.max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_ref,  cmap='gray', vmin=0, vmax=255)
axes[0].set_title("참조 이미지 (inclusion_1)", fontsize=13)
axes[0].axis('off')
axes[1].imshow(img_test, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("검사 이미지 (inclusion_2)", fontsize=13)
axes[1].axis('off')
plt.suptitle("오늘의 데이터: 두 이미지 비교 준비", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 1. absdiff — 두 이미지의 차이 계산

**`cv2.absdiff(src1, src2)`** 는 픽셀 단위로 `|src1 - src2|`를 계산합니다.

- **왜 이 함수가 필요한가?** 단순 빼기(`src1 - src2`)는 음수가 나오면 uint8 특성상 255로 감아버립니다(언더플로). 반면 `absdiff`는 순서에 관계없이 항상 올바른 절댓값 차이를 반환합니다.
- **두 입력의 조건**: shape과 dtype이 동일해야 합니다. 그레이스케일 vs BGR 혼합 시 오류.
- **제조 검사 연결**: 라인에서 "골든 샘플(정상품)"과 검사 대상을 픽셀 단위로 비교해 달라진 부분만 뽑아내는 **Comparative Inspection(비교 검사)** 의 핵심 연산입니다.

In [ ]:
diff = cv2.absdiff(img_ref, img_test)

print(f"absdiff 결과: shape={diff.shape}, dtype={diff.dtype}")
print(f"차이 범위: {diff.min()} ~ {diff.max()}, 평균: {diff.mean():.2f}")
print(f"차이 없는 픽셀(=0) 비율: {(diff == 0).sum() / diff.size * 100:.1f}%")
print(f"차이 큰 픽셀(>30) 비율: {(diff > 30).sum() / diff.size * 100:.1f}%")

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(img_ref, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("참조 이미지", fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_test, cmap='gray', vmin=0, vmax=255)
axes[1].set_title("검사 이미지", fontsize=12)
axes[1].axis('off')

axes[2].imshow(diff, cmap='hot', vmin=0, vmax=255)
axes[2].set_title(f"absdiff 결과\n(밝을수록 차이 큼, max={diff.max()})", fontsize=12)
axes[2].axis('off')

diff_enhanced = cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX)
axes[3].imshow(diff_enhanced, cmap='hot', vmin=0, vmax=255)
axes[3].set_title("absdiff (0~255 정규화)", fontsize=12)
axes[3].axis('off')

plt.suptitle("absdiff: inclusion_1 vs inclusion_2", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **차이 이미지(heatmap)**: 밝을수록 두 이미지 사이에 차이가 큰 픽셀 → 결함 위치 후보
- **차이 없는 픽셀 비율**: 70% 이상이면 배경(정상 배경) 영역이 대부분 같다는 의미
- **정규화 버전**: `diff`의 최대값이 작을 때(전체적으로 유사한 이미지) 시각적으로 차이를 확인하기 위해 0~255로 늘려줌
- 결함 위치와 차이가 큰 영역이 일치하는지 확인해보세요

> 🤔 **판단 질문 1:** 두 이미지가 같은 결함 유형(inclusion)인데 absdiff 결과에서 밝은 영역이 많다면, 그 의미는 두 결함의 **위치**가 다른 것일까요, **크기**가 다른 것일까요?

> 🤔 **판단 질문 2:** `diff.max()`가 255에 가까울수록 두 이미지가 더 많이 다르다고 말할 수 있을까요? `diff.mean()`과 `diff.max()` 중 어느 것이 전체적인 차이 정도를 더 잘 나타낼까요?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- **챌린지 1**: `img_ref - img_test`(일반 numpy 빼기)와 `cv2.absdiff(img_ref, img_test)`의 결과를 비교해보세요. 어느 쪽이 더 정확한 차이를 보여주나요? 왜 다를까요?
- **챌린지 2**: `cv2.absdiff(img_ref, img_ref)`(같은 이미지끼리 비교)의 결과는 어떻게 될까요? `diff.max()`는 얼마일까요?
- **챌린지 3**: `crazing_1.jpg`와 `crazing_2.jpg`로 바꿔보면 inclusion 비교보다 차이가 클까요 작을까요? (두 결함 유형의 패턴 차이를 생각해보세요)

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 2. addWeighted — 가중치 블렌딩

**`cv2.addWeighted(src1, alpha, src2, beta, gamma)`** 는 두 이미지를 가중치를 적용해 합성합니다.

- **수식**: `dst = alpha × src1 + beta × src2 + gamma`
- **투명도 효과**: `alpha=0.7, beta=0.3`이면 src1이 70%, src2가 30% 기여 → 자연스러운 블렌딩
- **제조 검사 연결**: 결함 위치를 파란색/빨간색으로 표시한 마스크를 원본 이미지 위에 반투명하게 오버레이할 때 사용합니다. "어디가 결함인지" 시각적으로 확인하는 표준 방법입니다.

In [ ]:
# absdiff 결과를 컬러(빨강)로 만들어 원본 위에 오버레이
diff_color = cv2.applyColorMap(diff_enhanced, cv2.COLORMAP_HOT)
img_ref_bgr = cv2.cvtColor(img_ref, cv2.COLOR_GRAY2BGR)

alpha_vals = [0.9, 0.7, 0.5, 0.3]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for col, alpha in enumerate(alpha_vals):
    beta = 1.0 - alpha
    blended = cv2.addWeighted(img_ref_bgr, alpha, diff_color, beta, 0)
    blended_rgb = cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)
    axes[col].imshow(blended_rgb)
    axes[col].set_title(f"alpha={alpha:.1f}, beta={beta:.1f}\n원본 {int(alpha*100)}% + 차이맵 {int(beta*100)}%",
                        fontsize=11)
    axes[col].axis('off')

plt.suptitle("addWeighted: 원본 + 차이 히트맵 오버레이 (alpha 변화)", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **alpha 높음(0.9)**: 원본이 주를 이루고 차이맵이 옅게 보임 → 원래 이미지 정보 유지
- **alpha 낮음(0.3)**: 차이맵이 강하게 보임 → 결함 위치 강조
- **실무 균형점**: 원본의 결함 문맥(형상, 위치)과 차이 정보를 동시에 보여주는 `alpha=0.6~0.7` 범위가 자주 쓰임
- 결함 주변의 빨간/주황 부분이 실제 결함 위치와 얼마나 일치하는지 확인해보세요

> 🤔 **판단 질문 1:** `alpha + beta = 1.0`이 되어야 자연스러운 이유는 뭘까요? `alpha=0.7, beta=0.7`로 설정하면 결과가 어떻게 변할까요?

> 🤔 **판단 질문 2:** `gamma` 값을 50으로 높이면 어떤 시각적 변화가 생길까요? 검사 리포트에서 이 기능이 유용한 경우는 언제일까요?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- **챌린지 1**: `gamma=50`으로 설정하면 이미지가 전체적으로 밝아지나요? 어떤 상황에서 gamma 오프셋이 유용할까요?
- **챌린지 2**: `alpha=1.5, beta=-0.5`처럼 합이 1이 아닌 경우를 시도해보세요. 어떤 결과가 나올까요? (힌트: 0~255 클리핑)
- **챌린지 3**: 결함 위치를 더 잘 강조하려면 alpha와 beta 중 어느 것을 높여야 할까요? 가장 적합한 비율을 직접 찾아보세요.

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 3. bitwise 연산 — AND, OR, NOT

bitwise 연산은 **마스크(0 또는 255로만 구성된 이진 이미지)** 와 함께 쓸 때 강력해집니다.

- **bitwise_AND**: 마스크가 255인 곳은 원본 유지, 0인 곳은 검정으로 만들기 → **관심 영역 추출**
- **bitwise_OR**: 두 마스크를 합치기 → **여러 결함 영역 통합**
- **bitwise_NOT**: 마스크 반전 → **배경 추출 (결함 제외한 나머지)**

**제조 검사 연결**: `threshold`로 만든 이진 마스크에 `bitwise_AND`를 적용해 결함 픽셀만 원본에서 추출하는 패턴은 W3부터 계속 반복됩니다.

In [ ]:
# absdiff 결과를 threshold로 이진화 → 마스크 생성
_, mask = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

mask_inv   = cv2.bitwise_not(mask)
img_defect = cv2.bitwise_and(img_test, img_test, mask=mask)
img_bg     = cv2.bitwise_and(img_test, img_test, mask=mask_inv)

print(f"마스크 통계: 결함 픽셀(255)={mask.sum()//255:,}개, 전체={mask.size:,}개")
print(f"결함 영역 비율: {mask.sum()//255 / mask.size * 100:.2f}%")

fig, axes = plt.subplots(2, 3, figsize=(20, 13))

axes[0, 0].imshow(diff, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title(f"absdiff 결과\n(원본 차이)", fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(mask, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title(f"threshold 마스크\n(차이>30인 영역=흰색)", fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(mask_inv, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title(f"bitwise_NOT 마스크\n(마스크 반전)", fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(img_test, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title("원본 검사 이미지", fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(img_defect, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title("bitwise_AND(원본, 마스크)\n→ 결함 영역만 추출", fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(img_bg, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title("bitwise_AND(원본, NOT마스크)\n→ 배경(결함 제외) 추출", fontsize=12)
axes[1, 2].axis('off')

plt.suptitle("bitwise 연산: 마스크로 결함 영역 추출", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **threshold 마스크**: 차이가 큰 픽셀(>30)만 흰색(255). 임계값 30은 조명 노이즈를 무시하기 위한 선택
- **bitwise_AND 결과(결함)**: 마스크 흰색 부분만 원본 픽셀이 살고, 나머지는 0(검정). 결함 위치만 뽑아냄
- **bitwise_AND 결과(배경)**: NOT 마스크 적용 → 결함 부분이 검정, 배경만 남음. "결함을 지운 이미지"
- `img_defect + img_bg`를 더하면 원본 `img_test`가 복원됨 (마스크가 완벽히 보완적이므로)

> 🤔 **판단 질문 1:** threshold 임계값을 30에서 10으로 낮추면 마스크에서 흰색 영역이 늘어날까요 줄어들까요? 낮은 임계값의 장단점은?

> 🤔 **판단 질문 2:** `bitwise_AND(img_defect, img_bg)`의 결과는 어떻게 될까요? 두 마스크의 관계를 생각해보세요.

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- **챌린지 1**: threshold 임계값을 `30`에서 `5`로 낮추면 결함 마스크 영역이 어떻게 변하나요? 너무 낮은 임계값의 문제점은?
- **챌린지 2**: `cv2.bitwise_or(mask, mask_inv)`의 결과를 계산해보세요. 결과가 전부 255인 이미지가 나올까요?
- **챌린지 3**: `bitwise_and` 대신 `bitwise_or(img_defect, img_bg)`를 계산하면 원본 `img_test`와 같아질까요? 직접 확인해보세요.

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 4. bitwise_OR — 두 마스크 합치기

`bitwise_OR`는 두 마스크를 **합집합**으로 합칩니다.

- 용도: 서로 다른 조건(예: 어두운 결함 마스크 + 밝은 결함 마스크)을 하나의 마스크로 통합할 때
- 또는 여러 결함 유형의 감지 결과를 합칠 때

**제조 검사 연결**: AOI에서 한 이미지에 여러 결함이 혼재할 때(예: pitted_surface에 scratches도 함께 있는 경우) 각각의 마스크를 `bitwise_OR`로 합쳐 전체 결함 지도(defect map)를 만듭니다.

In [ ]:
# 두 가지 임계값으로 마스크 생성 → OR로 합치기
_, mask_bright = cv2.threshold(diff, 20, 255, cv2.THRESH_BINARY)       # 약한 차이도 포함
_, mask_strong = cv2.threshold(diff, 60, 255, cv2.THRESH_BINARY)       # 강한 차이만

mask_combined = cv2.bitwise_or(mask_bright, mask_strong)

print(f"약한 차이 마스크(>20)  픽셀 수: {mask_bright.sum()//255:,}개 ({mask_bright.sum()//255/mask_bright.size*100:.1f}%)")
print(f"강한 차이 마스크(>60)  픽셀 수: {mask_strong.sum()//255:,}개 ({mask_strong.sum()//255/mask_strong.size*100:.1f}%)")
print(f"OR 결합 마스크         픽셀 수: {mask_combined.sum()//255:,}개 ({mask_combined.sum()//255/mask_combined.size*100:.1f}%)")
print(f"\n참고: OR 결합 == 약한 차이 마스크? {np.array_equal(mask_bright, mask_combined)}")

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

panels = [
    (mask_bright,   "threshold > 20\n(약한 차이 포함)",   'Blues'),
    (mask_strong,   "threshold > 60\n(강한 차이만)",       'Reds'),
    (mask_combined, "bitwise_OR 결합",                     'Greens'),
    (cv2.bitwise_and(img_test, img_test, mask=mask_combined),
     "OR 마스크로\n원본 추출",                              'gray'),
]

for col, (im, title, cmap) in enumerate(panels):
    axes[col].imshow(im, cmap=cmap, vmin=0, vmax=255)
    axes[col].set_title(title, fontsize=11)
    axes[col].axis('off')

plt.suptitle("bitwise_OR: 두 마스크 합치기", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **OR 결합 = 약한 마스크?**: 강한 차이(>60)는 반드시 약한 차이(>20)에도 포함되므로, OR 결과는 약한 마스크와 같아짐. `A ⊇ B` 이면 `A ∪ B = A`
- 이 실험은 **두 마스크의 포함 관계**를 확인하는 좋은 방법입니다
- **실무에서 유용한 OR**: 완전히 다른 두 조건(예: 어두운 결함 마스크 + 밝은 결함 마스크)일 때 합집합이 의미 있음

> 🤔 **판단 질문:** `bitwise_AND(mask_bright, mask_strong)`은 어떤 픽셀 집합을 나타낼까요? `mask_strong`과 같을까요?

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- **챌린지 1**: `cv2.bitwise_and(mask_bright, mask_strong)`을 계산하면 `mask_strong`과 동일한 결과가 나올까요? `np.array_equal()`로 확인해보세요.
- **챌린지 2**: 완전히 다른 결함 유형 두 장(예: `inclusion_1.jpg`와 `scratches_1.jpg`)에 각각 threshold를 적용해 두 마스크를 OR로 합쳐보세요. 어떤 의미인가요?
- **챌린지 3**: `bitwise_xor(mask_bright, mask_strong)` (XOR)는 어떤 영역을 나타낼까요? 실행 전에 예측해보세요.

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔧 5. 세 연산 통합 — 결함 감지 미니 파이프라인

오늘 배운 세 가지 연산을 **하나의 파이프라인**으로 연결합니다.

```
참조 이미지 + 검사 이미지
    ↓ absdiff
차이 이미지
    ↓ threshold
결함 마스크
    ↓ bitwise_AND
결함 영역 추출
    ↓ addWeighted
원본 + 결함 오버레이
```

이것이 **W6D4 차이 기반 검출**의 기초 버전입니다.

In [ ]:
def detect_diff_pipeline(img_ref, img_test, threshold_val=30, alpha=0.6):
    diff       = cv2.absdiff(img_ref, img_test)
    _, mask    = cv2.threshold(diff, threshold_val, 255, cv2.THRESH_BINARY)
    extracted  = cv2.bitwise_and(img_test, img_test, mask=mask)

    diff_color = cv2.applyColorMap(cv2.normalize(diff, None, 0, 255, cv2.NORM_MINMAX),
                                   cv2.COLORMAP_HOT)
    img_bgr    = cv2.cvtColor(img_test, cv2.COLOR_GRAY2BGR)
    overlay    = cv2.addWeighted(img_bgr, alpha, diff_color, 1 - alpha, 0)

    defect_pixels = int(mask.sum()) // 255
    defect_ratio  = defect_pixels / mask.size * 100
    return diff, mask, extracted, overlay, defect_pixels, defect_ratio


fig, big_axes = plt.subplots(3, 5, figsize=(26, 16))

test_pairs = [
    ("inclusion",       "inclusion_1",      "inclusion_2"),
    ("pitted_surface",  "pitted_surface_1", "pitted_surface_2"),
    ("scratches",       "scratches_1",      "scratches_2"),
]

col_titles = ["참조", "검사", "absdiff", "마스크", "오버레이"]
for col, t in enumerate(col_titles):
    big_axes[0, col].set_title(t, fontsize=13, fontweight='bold', pad=10)

for row, (defect, ref_name, test_name) in enumerate(test_pairs):
    src_ref  = cv2.imread(str(DATA_DIR / defect / f"{ref_name}.jpg"),  cv2.IMREAD_GRAYSCALE)
    src_test = cv2.imread(str(DATA_DIR / defect / f"{test_name}.jpg"), cv2.IMREAD_GRAYSCALE)

    diff, mask, extracted, overlay, n_px, ratio = detect_diff_pipeline(src_ref, src_test)

    big_axes[row, 0].imshow(src_ref,  cmap='gray', vmin=0, vmax=255)
    big_axes[row, 0].set_ylabel(defect.replace('_', '\n'), fontsize=11, fontweight='bold')
    big_axes[row, 0].axis('off')

    big_axes[row, 1].imshow(src_test, cmap='gray', vmin=0, vmax=255)
    big_axes[row, 1].axis('off')

    big_axes[row, 2].imshow(diff, cmap='hot', vmin=0, vmax=255)
    big_axes[row, 2].axis('off')

    big_axes[row, 3].imshow(mask, cmap='gray', vmin=0, vmax=255)
    big_axes[row, 3].set_xlabel(f"결함 픽셀: {n_px:,}개\n({ratio:.1f}%)", fontsize=10)
    big_axes[row, 3].axis('off')

    big_axes[row, 4].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    big_axes[row, 4].axis('off')

plt.suptitle("결함 감지 미니 파이프라인: absdiff → threshold → bitwise → addWeighted",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- **결함 픽셀 비율**: 같은 결함 유형이어도 두 이미지 간 결함 위치가 다르면 비율이 높아짐
- **pitted_surface**: 표면 전체에 작은 홈이 분산되어 있어 두 이미지 간 차이가 넓게 퍼짐
- **scratches**: 선형 결함의 방향이 달라지면 absdiff 결과가 두 선을 모두 포함
- **한계 관찰**: 같은 결함 유형인데 단순 픽셀 비교만으로는 "결함 있음"과 "결함 위치 다름"을 구분하지 못함 → W6에서 특징점 기반 정렬로 해결

> 🤔 **판단 질문 1:** 세 결함 유형 중 어느 것이 "위치가 다른 두 이미지" 비교에서 결함 픽셀 비율이 가장 높게 나왔나요? 왜 그 유형이 높을까요?

> 🤔 **판단 질문 2:** 이 파이프라인으로 실제 AOI 검사를 하려면 어떤 추가 조건이 필요할까요? (힌트: 두 이미지의 정렬 상태를 생각해보세요)

---

### 🔨 깨뜨리기 챌린지
다음 중 하나를 시도해보세요:
- **챌린지 1**: `threshold_val`을 5로 낮추면 결함 픽셀 비율이 크게 오르는 결함 유형은 무엇인가요? 노이즈 때문인지 실제 차이 때문인지 어떻게 구별할까요?
- **챌린지 2**: 같은 이미지를 참조와 검사로 동시에 넣으면(`img_ref`, `img_ref`) 결함 픽셀 비율은 얼마가 되어야 할까요? 직접 확인해보세요.
- **챌린지 3**: `alpha=0.6` → `alpha=0.0`으로 바꾸면 오버레이 결과가 어떻게 되나요? `alpha=1.0`이면?

예상 결과를 먼저 적어보고, 실행해서 비교해보세요.

## 🔬 파라미터 실험: threshold_val 변화에 따른 결함 감지 민감도

실제 검사에서 `threshold_val`(차이 감지 임계값)은 **핵심 튜닝 파라미터**입니다.

- 너무 낮으면 → 조명 노이즈, 미세한 밝기 차이까지 결함으로 오검(False Positive)
- 너무 높으면 → 실제 결함도 무시(False Negative)

**검사 정확도 관점**: 제조업에서는 오검과 누락 중 어느 것이 더 큰 손실인지 결정하고 임계값을 설정합니다.

In [ ]:
threshold_vals = [5, 10, 20, 30, 50, 80, 120, 180]
defect_names   = ["inclusion", "scratches", "pitted_surface"]

results = {d: [] for d in defect_names}

for defect in defect_names:
    r = cv2.imread(str(DATA_DIR / defect / f"{defect}_1.jpg"), cv2.IMREAD_GRAYSCALE)
    t = cv2.imread(str(DATA_DIR / defect / f"{defect}_2.jpg"), cv2.IMREAD_GRAYSCALE)
    d = cv2.absdiff(r, t)
    for th in threshold_vals:
        _, m = cv2.threshold(d, th, 255, cv2.THRESH_BINARY)
        results[defect].append(m.sum() // 255 / m.size * 100)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors_d = {'inclusion': '#e74c3c', 'scratches': '#9b59b6', 'pitted_surface': '#2ecc71'}

for defect, vals in results.items():
    axes[0].plot(threshold_vals, vals, marker='o', linewidth=2,
                 color=colors_d[defect], label=defect)

axes[0].set_title("threshold_val 변화에 따른 결함 픽셀 비율", fontsize=13)
axes[0].set_xlabel("threshold_val", fontsize=11)
axes[0].set_ylabel("결함 픽셀 비율 (%)", fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)
axes[0].axvline(x=30, color='gray', linestyle='--', alpha=0.7, label='기본값=30')

ref_inc  = cv2.imread(str(DATA_DIR / "inclusion"      / "inclusion_1.jpg"),      cv2.IMREAD_GRAYSCALE)
tst_inc  = cv2.imread(str(DATA_DIR / "inclusion"      / "inclusion_2.jpg"),      cv2.IMREAD_GRAYSCALE)
ref_scr  = cv2.imread(str(DATA_DIR / "scratches"      / "scratches_1.jpg"),      cv2.IMREAD_GRAYSCALE)
tst_scr  = cv2.imread(str(DATA_DIR / "scratches"      / "scratches_2.jpg"),      cv2.IMREAD_GRAYSCALE)

diff_inc = cv2.absdiff(ref_inc, tst_inc)
diff_scr = cv2.absdiff(ref_scr, tst_scr)

hist_inc = cv2.calcHist([diff_inc], [0], None, [256], [0, 256])
hist_scr = cv2.calcHist([diff_scr], [0], None, [256], [0, 256])

axes[1].plot(range(256), hist_inc.flatten(), color='#e74c3c', linewidth=1.5,
             label='inclusion diff 히스토그램')
axes[1].plot(range(256), hist_scr.flatten(), color='#9b59b6', linewidth=1.5,
             label='scratches diff 히스토그램')
axes[1].axvline(x=30, color='gray',   linestyle='--', alpha=0.8, label='threshold=30')
axes[1].axvline(x=60, color='orange', linestyle='--', alpha=0.8, label='threshold=60')
axes[1].set_title("absdiff 결과의 픽셀 분포 (히스토그램)", fontsize=13)
axes[1].set_xlabel("차이 강도 (0~255)", fontsize=11)
axes[1].set_ylabel("픽셀 수", fontsize=11)
axes[1].legend(fontsize=10)
axes[1].set_xlim([0, 130])
axes[1].grid(alpha=0.3)

plt.suptitle("threshold_val 튜닝: 결함 감지 민감도 분석", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 threshold=30 기준 결함 픽셀 비율:")
for defect in defect_names:
    idx = threshold_vals.index(30)
    print(f"  {defect:20s}: {results[defect][idx]:.2f}%")

### 해석 가이드
- **급격히 꺾이는 구간**: 그래프에서 결함 픽셀 비율이 급감하는 threshold_val 근처가 노이즈와 실제 차이의 경계
- **히스토그램 피크 위치**: diff 히스토그램에서 대부분 픽셀이 0~10 구간에 몰려 있다면 두 이미지가 전반적으로 유사
- **결함 유형별 차이**: pitted_surface는 전체 표면에 분산 → 낮은 threshold에서도 높은 비율
- **실무 권장**: 「노이즈 기여분 < 임계값 < 가장 작은 결함의 차이값」 범위에서 설정

> 🤔 **판단 질문:** 히스토그램에서 0 주변(차이 없음)의 픽셀 수가 압도적으로 많고 그 다음 피크가 40~80 사이에 있다면, threshold를 어디에 설정하는 게 좋을까요?

## 📝 정리

### 오늘 배운 것
| 함수 | 핵심 포인트 |
|------|-------------|
| `cv2.absdiff(src1, src2)` | 픽셀 단위 절댓값 차이. 두 이미지가 다른 곳 → 밝게, 같은 곳 → 0(검정) |
| `cv2.addWeighted(src1, α, src2, β, γ)` | 가중치 블렌딩. `α×src1 + β×src2 + γ`, 결과 0~255 클리핑 |
| `cv2.bitwise_and(src1, src2, mask)` | 비트 AND. mask=255인 곳만 원본 유지 → **관심 영역 추출** |
| `cv2.bitwise_or(src1, src2)` | 비트 OR. 두 마스크 합집합 → **여러 결함 영역 통합** |
| `cv2.bitwise_not(src)` | 비트 NOT. 픽셀값 반전(0↔255) → **마스크 반전** |

### 파이프라인 개념 정립
```
참조 이미지 + 검사 이미지
    ↓ absdiff           ← W1D5 오늘 배운 연산
차이 이미지
    ↓ threshold         ← W3D1에서 심화
이진 마스크
    ↓ bitwise_AND       ← W1D5 오늘 배운 연산
결함 영역 추출
    ↓ addWeighted       ← W1D5 오늘 배운 연산
원본 + 오버레이 시각화
```

### uint8 연산 주의사항
| 상황 | 잘못된 방법 | 올바른 방법 |
|------|------------|-------------|
| 두 이미지 차이 | `img1 - img2` (언더플로 가능) | `cv2.absdiff(img1, img2)` |
| 두 이미지 합성 | `img1 + img2` (오버플로 가능) | `cv2.addWeighted(img1, α, img2, β, 0)` |
| 마스크 적용 | `img * (mask / 255)` | `cv2.bitwise_and(img, img, mask=mask)` |

### 복습 퀴즈
1. `cv2.absdiff(a, b)`와 `np.abs(a.astype(int) - b.astype(int)).astype(np.uint8)`의 결과는 같을까요? 왜 그런가요?
2. `addWeighted`에서 `alpha + beta > 1.0`이면 어떤 현상이 생기나요?
3. `bitwise_and(img, img, mask=mask)`에서 첫 번째와 두 번째 `img`에 **다른 이미지**를 넣으면 어떻게 될까요?
4. `bitwise_not`을 두 번 연속 적용(`bitwise_not(bitwise_not(img))`)하면 원본과 같아질까요?
5. 이 파이프라인의 가장 큰 한계는 무엇이고, W6에서 어떻게 해결할 예정인가요?

### 다음에 할 것 (W1 Summary)
- W1 전체 복습: `imread` → `cvtColor` → `calcHist` → `CLAHE` → `absdiff/bitwise` 연결
- 6종 결함 전체에 W1 파이프라인 적용해 정리